# Flash attention
## 搭建环境
在ubuntu虚拟机上搭建了环境，可以正确编译以及cuda程序，实现cuda程序的自动跳转，方便编写cuda代码
因此编写代码就放在虚拟机上

因为flash attention2 3 需要利用特殊的硬件特性，因此需要在租相应的平台，在平台上搭建环境很麻烦

先看看autodl，能不能采用docker的方式运行
## 实现flash attention
### 实现层次
需要先看看怎么实现，应该是直接实现flash attention 算子，包括对应的forward、backward

上层的transfomer 模块相应的调用算子（看一下以前的transformer实现）

最后确认了实现的层次：Ops -> CUDA Kernel -> Pybind -> Op Wrapper -> Module Call
### 实现方法
应该用什么实现呢？cuda、triton、cuTile

使用cuda编程的话，应该借用cutblass模版库进行编程

如果使用triton实现的话，绕过了pybind，但是将cuda后端管理的ptr给triton模块，然后再在triton层面进行编程

使用cuTile编程的话，依旧是python编程，和triton类似

### 测试程序
需要先编写测试程序，首先是flash attention 算子的测试程序，该测试程序需要包括一下几个方面

1.直接调用cuda程序中的实现进行测试（确定在ops层面进行测试）

2.需要验证算子的正确性，那和什么进行比对呢（或者在python层面进行比对正确性，参考一下以前的比较）

3.每个测试应该进行多次迭代，从而可以测试时间，包括不同seq长度

（可以参考官方的flash attention实现，看看它是怎么进行测试的）





接下来要做的事：

在autodl平台上跑一次

了解cuTile，决定是用cutlass还是用cuTile

看看别人的实现以及怎么测试、怎么benchmark

backward 不应该都是tensor操作吗，直接调用Narray api不就构建不了计算图了吗？（需要构建一个ops用来计算backward）

测试的时候是调用ops测试，还是module呢，以及怎么获得时间


In [ ]:
!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git

# Download the PTB dataset

import urllib.request
import os

!mkdir -p './data/ptb'
# Download Penn Treebank dataset
ptb_data = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb."
for f in ['train.txt', 'test.txt', 'valid.txt']:
    if not os.path.exists(os.path.join('./data/ptb', f)):
        urllib.request.urlretrieve(ptb_data + f, os.path.join('./data/ptb', f))

In [4]:
!make

-- Found pybind11: /home/xyx/needle_env/lib/python3.12/site-packages/pybind11/include (found version "3.0.1")
-- Found cuda, building cuda backend
-- Building CUDA backend with FlashAttention stub support
-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /home/xyx/needle/build
make[1]: Entering directory '/home/xyx/needle/build'
make[2]: Entering directory '/home/xyx/needle/build'
make[3]: Entering directory '/home/xyx/needle/build'
make[3]: Leaving directory '/home/xyx/needle/build'
[ 50%] Built target ndarray_backend_cpu
make[3]: Entering directory '/home/xyx/needle/build'
[ 75%] Building NVCC (Device) object CMakeFiles/ndarray_backend_cuda.dir/src/ndarray_backend_cuda_generated_ndarray_backend_cuda.cu.o
/home/xyx/needle/src/ndarray_backend_cuda.cu(661): warning #177-D: variable "to_tf32" was declared but never referenced
    cutlass::NumericConverter<cutlass::tfloat32_t, float> to_tf32;
                                                         

In [ ]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

In [ ]:
import sys
sys.path.append('./python')

## 测试
### 正确性验证
如果在NDArray层面去测试的话，需要借用numpy来进行比较，而numpy没有现有的api计算flash attentionn，需要根据已知的qkv计算self attention，因此需要创建numpy的qkv，然后根据self attention的定义去计算，再与cuda 的NDArray计算结果进行比较，比较麻烦。

因此本测试在flash attention module层面进行正确性验证，仿照已有测试中的attention_activation 测试编写，将flash attention的结果与已知的label进行对比，同时编写了新的测试将结果和torch的flash attention 计算结果对比，在与torch对比的测试用例 sequence length比较长，符合实际。
### benchmark
对于benchmark说，需要计算TFLOPS，为了最贴近计算，采用算子层面进行benchmark，先对GPU进行预热，flashattention进行计算，迭代50次，计算时间以及总的操作数，从而计算TFOPS。当前的计算基于causal = false，dropout = 0

In [ ]:
!python3 -m pytest tests/hw4/test_transformer.py -l -v -k "attention_activation_vs_torch"

In [ ]:
# 正确性验证
#!python3 -m pytest tests/project/test_flashattention.py -l -v -k "test_flashattention_activation and False and 0.0"
!python3 -m pytest tests/project/test_flashattention.py -l -v -k "test_attention_activation_vs_torch and False and 0.0"

In [ ]:
!python3 -m pytest tests/project/test_flashattention.py -l -v -k "test_attention_activation_vs_torch and 64 and False and 0.0 and cuda"

In [ ]:
# stub hook 验证
import sys
!{sys.executable} -m pytest tests/project/test_flashattention_stub.py -l -v

In [ ]:
# 计算TFLOPS
!python3 tests/project/benchmark.py --batch_size 8 --num_heads 12 --seq_len 1024 --head_dim 64 --dropout 0.0

In [ ]:
# baseline
!python3 tests/project/benchmark_torch.py --batch_size 8 --num_heads 12 --seq_len 1024 --head_dim 64 --dropout 0.0

In [ ]:
# ncu profile
!ncu --launch-skip 10 --launch-count 1

## stub
对于ops层面的stub来说，是返回tensor tuple还是tensor呢？
* 需要查看probs对于后续有没有什么作用，如果没有作用可以直接返回result tensor
* 如果返回tensor tuple，是backend计算直接返回tensor tuple还是在ops层进行组装后返回，需要参考stack的实现，返回tensor tuple后进行第i个结果的取用会不会产生额外的开销

最后决定只是返回result，因此定义为tensorops


## 调用层次关系
module调用ops进行计算，ops的forward计算的是NDArray，使用NDArray定义的array api进行计算，NDArray的backend device不同，因此调用进不同的device function。

## 精度
在pytorch中的attention的实现中，因为A100的tensor core只接受输入的形式为 fp16/bf16/tf32，不支持fp32的输入，因此没有开启混合精度的情况下，是没有使用flash attention的实现的，而使用的是原生的的其他优化过的kernel

如果开启混合精度模式，计算的流程如下：QKV（fp32）加载到sram中cast 为（fp16），使用tensor做矩阵乘法时，结果accumulator保存为fp32。进行softmax的时候，为了防止溢出保持fp32，再cast为fp16与 V（fp16）相乘，
最后的结果O为fp16（为了节省sram 写到 GMEM中的带宽）。

现在我自己的实现最好确定输入的类型时fp16，那这样其他实现的cuda计算就用不了了，一个weired解决办法是输入继续保持为fp32但是计算的结果o为fp32再重新cast为fp16。

* 看看别人的flash attention怎么实现的
    * 别人的flash attention中没有使用cute模版编程，只是借用了cutlass的gemm实现来优化自己的设计。

* 如果在我的实现中使用cutlass，应该怎么使用，和cute有区别吗？
    * CuTe 强大的 Layout Algebra (布局代数) 能够让你在不陷入指针算术泥潭的情况下，优雅地处理复杂的 Tensor Core 数据映射、Shared Memory Swizzle 和 Bank Conflict。


1. 现在需要研究cute 的核心，使用cute 模版编程来实现自己的flash attention。


